#### Task 2: Contract type vs Churn - count of churned customers for each contract (PivotTable equivalent).

In [2]:
import os
import numpy as np
import pandas as pd
 
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
CANON = ["CustomerID", "Gender", "SeniorCitizen", "Partner", "Dependents", "Tenure", "PhoneService",
         "InternetService", "Contract", "MonthlyCharges", "TotalCharges", "Churn"]
 
 
def load_data():
    """Load the real Telco-Customer-Churn.csv if it is in this folder, otherwise the simulated sample."""
    for name in ("Telco-Customer-Churn.csv", "WA_Fn-UseC_-Telco-Customer-Churn.csv", "Telco-Customer-Churn_sample.csv"):
        path = os.path.join(HERE, name)
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"Loaded {name}: {len(df):,} rows" + ("  (SIMULATED STAND-IN DATA, not the real file)" if "sample" in name else ""))
            break
    else:
        raise FileNotFoundError("Put Telco-Customer-Churn.csv (or run generate_sample_data.py) in the same folder.")
    df = df.rename(columns={c: next((k for k in CANON if k.lower() == c.lower()), c) for c in df.columns})   # customerID -> CustomerID, tenure -> Tenure
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")        # blank strings become NaN
    return df
 
 
df = load_data()
 
# Excel equivalent: PivotTable, Rows = Contract, Columns = Churn, Values = Count of CustomerID
pivot = pd.pivot_table(df, index="Contract", columns="Churn", values="CustomerID", aggfunc="count", margins=True, margins_name="Total")
pivot = pivot.loc[["Month-to-month", "One year", "Two year", "Total"]]
print("\nCustomer counts by Contract and Churn:")
print(pivot.to_string())
 
churned = pivot["Yes"].drop("Total")
print("\nChurned customers per contract type:")
print(churned.to_string())
 
rate = (pivot["Yes"] / pivot["Total"] * 100).round(1)
print("\nChurn rate within each contract type (%):")
print(rate.to_string())
print(f"\nHighest churn: {rate.drop('Total').idxmax()} ({rate.drop('Total').max()}%); lowest: {rate.drop('Total').idxmin()} ({rate.drop('Total').min()}%)")

Loaded Telco-Customer-Churn_sample.csv: 2,000 rows  (SIMULATED STAND-IN DATA, not the real file)

Customer counts by Contract and Churn:
Churn             No  Yes  Total
Contract                        
Month-to-month   594  519   1113
One year         353   53    406
Two year         459   22    481
Total           1406  594   2000

Churned customers per contract type:
Contract
Month-to-month    519
One year           53
Two year           22

Churn rate within each contract type (%):
Contract
Month-to-month    46.6
One year          13.1
Two year           4.6
Total             29.7

Highest churn: Month-to-month (46.6%); lowest: Two year (4.6%)
